# Week 06 — Generalization and Historical Baseline

## Objective

Evaluate the generalization of the existing LogisticRegression
pipeline and compare it against a historical KSI-frequency baseline.

## 1. Load W05 Modeling Dataset

In [1]:
from pathlib import Path

import pandas as pd

data_path = Path(
    "../data/processed/intersection_day_2018_w5_features.csv"
)

model_df = pd.read_csv(
    data_path,
    parse_dates=["prediction_date", "target_date"]
)

print("Dataset shape:", model_df.shape)
display(model_df.head(3))

Dataset shape: (823217, 8)


,intersection_id,prediction_date,target_date,historical_ksi_count,historical_traffic_volume,day_of_week,month,target
0,3,2018-01-01,2018-01-02,0,12960.0,1,1,0
1,3,2018-01-02,2018-01-03,0,12960.0,2,1,0
2,3,2018-01-03,2018-01-04,0,12960.0,3,1,0


In [2]:
feature_cols = [
    "historical_ksi_count",
    "historical_traffic_volume",
    "day_of_week",
    "month"
]

train_mask = model_df["prediction_date"] <= "2018-09-29"

val_mask = model_df["prediction_date"] >= "2018-10-01"

X_train = model_df.loc[train_mask, feature_cols]
y_train = model_df.loc[train_mask, "target"]

X_val = model_df.loc[val_mask, feature_cols]
y_val = model_df.loc[val_mask, "target"]

print("Train:", X_train.shape, "Positives:", y_train.sum())
print("Validation:", X_val.shape, "Positives:", y_val.sum())

Train: (613223, 4) Positives: 81
Validation: (207736, 4) Positives: 38


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Numerical preprocessing
numeric_features = [
    "historical_ksi_count",
    "historical_traffic_volume"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 2. Categorical preprocessing
categorical_features = ["day_of_week", "month"]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 3. Complete ML pipeline
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

# 4. Train model
model.fit(X_train, y_train)

# 5. Predict probabilities
train_proba = model.predict_proba(X_train)[:, 1]
val_proba = model.predict_proba(X_val)[:, 1]

# 6. Evaluate generalization
train_ap = average_precision_score(y_train, train_proba)
val_ap = average_precision_score(y_val, val_proba)

print(f"Train AP: {train_ap:.6f}")
print(f"Validation AP: {val_ap:.6f}")
print(f"Train prevalence: {y_train.mean():.6f}")
print(f"Validation prevalence: {y_val.mean():.6f}")

Train AP: 0.000784
Validation AP: 0.000372
Train prevalence: 0.000132
Validation prevalence: 0.000183


In [4]:
small_train_df = (
    model_df.loc[train_mask]
    .groupby("target", group_keys=False)
    .sample(frac=0.5, random_state=42)
)

X_small = small_train_df[feature_cols]
y_small = small_train_df["target"]

print("Small train:", X_small.shape)
print("Small train positives:", y_small.sum())

Small train: (306611, 4)
Small train positives: 40


In [5]:
from sklearn.base import clone

small_model = clone(model)

small_model.fit(X_small, y_small)

small_proba = small_model.predict_proba(X_val)[:, 1]

small_ap = average_precision_score(y_val, small_proba)

print(f"50% Train AP (validation): {small_ap:.6f}")
print(f"100% Train AP (validation): {val_ap:.6f}")

50% Train AP (validation): 0.000357
100% Train AP (validation): 0.000372


In [6]:
train_history = model_df.loc[
    train_mask,
    ["intersection_id", "target"]
]

intersection_risk = (
    train_history
    .groupby("intersection_id")["target"]
    .mean()
)

global_risk = y_train.mean()

baseline_val_proba = (
    model_df.loc[val_mask, "intersection_id"]
    .map(intersection_risk)
    .fillna(global_risk)
)

print("Baseline predictions:", len(baseline_val_proba))
print("Global historical risk:", global_risk)

Baseline predictions: 207736
Global historical risk: 0.00013208897905003564


In [9]:
baseline_ap = average_precision_score(y_val, baseline_val_proba)

print(baseline_ap)
print(val_ap)

0.00018173145349474596
0.0003719788062030399


In [10]:
import numpy as np

from sklearn.model_selection import TimeSeriesSplit

train_dates = model_df.loc[X_train.index, "prediction_date"]
unique_dates = pd.Index(train_dates.unique()).sort_values()

tscv = TimeSeriesSplit(n_splits=3, gap=1)

cv_splits = []

for train_days_idx, valid_days_idx in tscv.split(unique_dates):
    train_days = unique_dates[train_days_idx]
    valid_days = unique_dates[valid_days_idx]

    train_idx = np.flatnonzero(
        train_dates.isin(train_days).to_numpy()
    )

    valid_idx = np.flatnonzero(
        train_dates.isin(valid_days).to_numpy()
    )

    cv_splits.append((train_idx, valid_idx))

for i, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print(
        f"Fold {i}: "
        f"train positives={y_train.iloc[train_idx].sum()}, "
        f"validation positives={y_train.iloc[valid_idx].sum()}"
    )

Fold 1: train positives=14, validation positives=20
Fold 2: train positives=34, validation positives=18
Fold 3: train positives=51, validation positives=29


In [11]:
cv_history = model_df.loc[
    X_train.index,
    ["intersection_id", "target"]
]

fold_baseline_predictions = []

for train_idx, valid_idx in cv_splits:
    fold_train = cv_history.iloc[train_idx]
    fold_val = cv_history.iloc[valid_idx]

    fold_risk = (
        fold_train
        .groupby("intersection_id")["target"]
        .mean()
    )

    predictions = (
        fold_val["intersection_id"]
        .map(fold_risk)
        .fillna(fold_train["target"].mean())
    )

    fold_baseline_predictions.append(
        (fold_val["target"], predictions)
    )

In [12]:
baseline_fold_scores = []

for i, (y_true, predictions) in enumerate(
    fold_baseline_predictions,
    start=1
):
    ap = average_precision_score(y_true, predictions)

    baseline_fold_scores.append(ap)

    print(f"Fold {i} Baseline AP: {ap:.6f}")

baseline_mean_ap = np.mean(baseline_fold_scores)

print(f"\nMean Baseline AP: {baseline_mean_ap:.6f}")
print("Mean LogisticRegression AP: 0.000626")

Fold 1 Baseline AP: 0.000131
Fold 2 Baseline AP: 0.000200
Fold 3 Baseline AP: 0.000189

Mean Baseline AP: 0.000173
Mean LogisticRegression AP: 0.000626


## Week 06 — Results and Conclusions

### Objective

Evaluate the generalization of the existing LogisticRegression pipeline and compare its performance against a historical KSI-frequency baseline.

### 1. Generalization Analysis

| Metric | Train | Validation |
|---|---:|---:|
| Average Precision | 0.000784 | 0.000372 |
| Positive samples | 81 | 38 |

Training performance is approximately 2.1 times higher than validation performance, indicating a generalization gap.

This may suggest overfitting, but the extremely small number of positive samples makes the results unstable. High variance cannot be conclusively established from this comparison alone.

### 2. Effect of Training Sample Size

| Training Size | Validation AP |
|---|---:|
| 50% | 0.000357 |
| 100% | 0.000372 |

Increasing the training sample size slightly improved validation performance.

However, the difference is small, and the experiment used only one randomly selected subset. Therefore, it does not establish a reliable relationship between training size and model performance.

### 3. Historical Baseline vs LogisticRegression

A historical baseline was constructed using the observed KSI frequency for each intersection in the training data.

Both approaches were evaluated using the same three temporal cross-validation folds.

| Fold | Historical Baseline AP | LogisticRegression AP |
|---|---:|---:|
| 1 | 0.000131 | 0.000492 |
| 2 | 0.000200 | 0.000420 |
| 3 | 0.000189 | 0.000966 |
| **Mean** | **0.000173** | **0.000626** |

LogisticRegression outperformed the historical baseline on all three folds, achieving approximately **3.6 times higher mean Average Precision**.

### 4. Final Conclusions

- LogisticRegression demonstrates better risk-ranking performance than the historical KSI-frequency baseline under the current evaluation protocol.
- A noticeable train-validation performance gap suggests possible generalization difficulties.
- Increasing training size produced only a small improvement in validation performance.
- Model performance varies across temporal folds, indicating instability.
- The extremely rare positive class limits the reliability of the estimated metrics.
- Results remain exploratory because the dataset covers only the 2018 pilot period, event matching is incomplete, and no untouched final test has been evaluated.

**Overall conclusion:** LogisticRegression provides a measurable improvement over the historical baseline, but the current evidence is insufficient to claim robust generalization or readiness for real-world deployment.

Further validation using broader historical data and a strictly separated final test period is required.